# MathModelAgent - 智能数学建模助手

## 项目简介

这是一个基于HelloAgents框架的智能数学建模助手，能够帮助用户完成数学建模的全过程。

### 核心功能

- 多模型智能调度：根据任务难度选择不同模型
- RAG知识库：从本地知识库检索建模方法、代码模板
- HIL人机协作：关键节点暂停等待用户审批
- 联网搜索：获取最新的建模方法和论文
- LaTeX论文生成：自动生成符合格式的数学建模论文
- 可行性检查：技术、数据、时间、资源可行性评估

### 作者信息

- 姓名：16deng
- GitHub：[@16deng](https://github.com/16deng)
- 日期：2026-08-22

## 第1部分：环境配置

In [ ]:
# 安装依赖
!pip install -q hello-agents[all] langchain faiss-cpu sentence-transformers
!pip install -q pandas numpy matplotlib seaborn plotly
!pip install -q requests beautifulsoup4 google-search-results
!pip install -q jinja2 pdflatex python-dotenv tqdm pydantic

In [ ]:
# 导入必要的库
import os
import json
from dotenv import load_dotenv
from typing import Dict, Any, List, Optional

# HelloAgents
from hello_agents import SimpleAgent, HelloAgentsLLM, ToolRegistry
from hello_agents.tools import Tool, ToolParameter

# RAG知识库
from langchain_community.vectorstores import FAISS
from langchain_community.embeddings import HuggingFaceEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import DirectoryLoader, TextLoader

# 数据分析
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 联网搜索
import requests
from bs4 import BeautifulSoup

# LaTeX生成
from jinja2 import Template

# 代码执行
from src.code_executor import CodeExecutor, SafeCodeExecutor

# 加载环境变量
load_dotenv()

In [ ]:
# 配置LLM参数
# 请根据实际情况修改以下配置
os.environ["LLM_MODEL_ID"] = "Qwen/Qwen2.5-72B-Instruct"
os.environ["LLM_API_KEY"] = "your_api_key_here"
os.environ["LLM_BASE_URL"] = "https://api-inference.modelscope.cn/v1/"
os.environ["LLM_TIMEOUT"] = "60"

## 第2部分：RAG知识库

In [ ]:
class RAGKnowledgeBase:
    """RAG知识库类"""
    
    def __init__(self, knowledge_dir: str = "./knowledge"):
        """
        初始化RAG知识库
        
        Args:
            knowledge_dir: 知识库目录
        """
        self.knowledge_dir = knowledge_dir
        self.embeddings = HuggingFaceEmbeddings(
            model_name="sentence-transformers/all-MiniLM-L6-v2"
        )
        self.vectorstore = None
        self._load_knowledge()
    
    def _load_knowledge(self):
        """加载知识库"""
        if not os.path.exists(self.knowledge_dir):
            os.makedirs(self.knowledge_dir)
            print(f"知识库目录已创建: {self.knowledge_dir}")
            return
        
        # 加载文档
        loader = DirectoryLoader(
            self.knowledge_dir,
            glob="**/*.md",
            loader_cls=TextLoader
        )
        documents = loader.load()
        
        if not documents:
            print("知识库为空，请添加知识文档")
            return
        
        # 文本分割
        text_splitter = RecursiveCharacterTextSplitter(
            chunk_size=1000,
            chunk_overlap=200
        )
        texts = text_splitter.split_documents(documents)
        
        # 创建向量存储
        self.vectorstore = FAISS.from_documents(texts, self.embeddings)
        print(f"知识库加载完成，共 {len(texts)} 个文档片段")
    
    def search(self, query: str, k: int = 3) -> List[str]:
        """
        搜索知识库
        
        Args:
            query: 查询文本
            k: 返回结果数量
        
        Returns:
            相关文档列表
        """
        if not self.vectorstore:
            return ["知识库为空，请先添加知识文档"]
        
        results = self.vectorstore.similarity_search(query, k=k)
        return [doc.page_content for doc in results]
    
    def add_document(self, content: str, filename: str):
        """
        添加文档到知识库
        
        Args:
            content: 文档内容
            filename: 文件名
        """
        filepath = os.path.join(self.knowledge_dir, filename)
        with open(filepath, 'w', encoding='utf-8') as f:
            f.write(content)
        
        # 重新加载知识库
        self._load_knowledge()
        print(f"文档已添加: {filepath}")

## 第3部分：多模型调度

In [ ]:
class ModelScheduler:
    """多模型调度器"""
    
    def __init__(self):
        """初始化模型调度器"""
        self.models = {
            "simple": {
                "name": "Qwen/Qwen2.5-7B-Instruct",
                "description": "简单任务模型，节省token"
            },
            "medium": {
                "name": "Qwen/Qwen2.5-32B-Instruct",
                "description": "中等任务模型，平衡效果和成本"
            },
            "complex": {
                "name": "Qwen/Qwen2.5-72B-Instruct",
                "description": "复杂任务模型，保证质量"
            }
        }
        self.current_model = "medium"
    
    def select_model(self, task_difficulty: str) -> str:
        """
        根据任务难度选择模型
        
        Args:
            task_difficulty: 任务难度 (simple/medium/complex)
        
        Returns:
            模型名称
        """
        if task_difficulty in self.models:
            self.current_model = task_difficulty
            return self.models[task_difficulty]["name"]
        else:
            print(f"未知难度: {task_difficulty}，使用默认模型")
            return self.models[self.current_model]["name"]
    
    def get_current_model(self) -> str:
        """
        获取当前模型
        
        Returns:
            当前模型名称
        """
        return self.models[self.current_model]["name"]

## 第4部分：HIL人机协作

In [ ]:
class HILCollaboration:
    """HIL人机协作类"""
    
    def __init__(self):
        """初始化HIL协作"""
        self.actions = {
            "confirm": "确认当前结果",
            "edit": "编辑修改当前结果",
            "regenerate": "重新生成结果",
            "ask": "向AI提问获取更多信息",
            "skip": "跳过当前步骤",
            "abort": "中止整个流程"
        }
    
    def pause_for_review(self, step_name: str, content: str) -> str:
        """
        暂停等待用户审批
        
        Args:
            step_name: 步骤名称
            content: 当前内容
        
        Returns:
            用户选择的动作
        """
        print(f"\n{'='*50}")
        print(f"步骤: {step_name}")
        print(f"{'='*50}")
        print(f"\n当前结果:")
        print(content)
        print(f"\n可用动作:")
        for action, desc in self.actions.items():
            print(f"  - {action}: {desc}")
        
        while True:
            choice = input("\n请选择动作 (输入动作名称): ").strip().lower()
            if choice in self.actions:
                return choice
            else:
                print(f"无效选择，请输入: {', '.join(self.actions.keys())}")
    
    def get_user_input(self, prompt: str) -> str:
        """
        获取用户输入
        
        Args:
            prompt: 提示信息
        
        Returns:
            用户输入
        """
        return input(prompt).strip()

## 第5部分：联网搜索

In [ ]:
class WebSearch:
    """联网搜索类"""
    
    def __init__(self, api_key: Optional[str] = None):
        """
        初始化联网搜索
        
        Args:
            api_key: 搜索API密钥
        """
        self.api_key = api_key or os.getenv("SEARCH_API_KEY")
    
    def search(self, query: str, num_results: int = 5) -> List[Dict[str, str]]:
        """
        搜索网页
        
        Args:
            query: 搜索查询
            num_results: 返回结果数量
        
        Returns:
            搜索结果列表
        """
        # 这里使用示例实现，实际需要接入搜索API
        print(f"搜索: {query}")
        
        # 示例结果
        results = [
            {
                "title": f"搜索结果 {i+1}",
                "url": f"https://example.com/{i+1}",
                "snippet": f"这是关于 {query} 的搜索结果 {i+1}"
            }
            for i in range(num_results)
        ]
        
        return results
    
    def extract_content(self, url: str) -> str:
        """
        提取网页内容
        
        Args:
            url: 网页URL
        
        Returns:
            网页内容
        """
        try:
            response = requests.get(url, timeout=10)
            soup = BeautifulSoup(response.text, 'html.parser')
            
            # 提取正文
            paragraphs = soup.find_all('p')
            content = '\n'.join([p.get_text() for p in paragraphs])
            
            return content[:1000]  # 限制长度
        except Exception as e:
            return f"提取失败: {str(e)}"

## 第6部分：LaTeX论文生成

In [ ]:
class LatexGenerator:
    """LaTeX论文生成器"""
    
    def __init__(self, template_dir: str = "./templates"):
        """
        初始化LaTeX生成器
        
        Args:
            template_dir: 模板目录
        """
        self.template_dir = template_dir
        self.template = self._load_template()
    
    def _load_template(self) -> Template:
        """
        加载LaTeX模板
        
        Returns:
            Jinja2模板对象
        """
        # 默认模板
        template_str = r"""
\documentclass{article}
\usepackage[utf8]{inputenc}
\usepackage{amsmath}
\usepackage{amsfonts}
\usepackage{amssymb}
\usepackage{graphicx}
\usepackage{hyperref}

\title{ {{ title }} }
\author{ {{ author }} }
\date{ \today }

\begin{document}

\maketitle

\begin{abstract}
{{ abstract }}
\end{abstract}

\section{问题重述}
{{ problem_restatement }}

\section{问题分析}
{{ problem_analysis }}

\section{模型假设}
{{ model_assumptions }}

\section{符号说明}
{{ symbol_description }}

\section{模型建立与求解}
{{ model_establishment }}

\section{模型验证}
{{ model_verification }}

\section{模型评价与改进}
{{ model_evaluation }}

\section{参考文献}
{{ references }}

\end{document}
"""
        return Template(template_str)
    
    def generate(self, content: Dict[str, str], output_path: str):
        """
        生成LaTeX文件
        
        Args:
            content: 论文内容
            output_path: 输出路径
        """
        # 渲染模板
        latex_content = self.template.render(**content)
        
        # 保存文件
        with open(output_path, 'w', encoding='utf-8') as f:
            f.write(latex_content)
        
        print(f"LaTeX文件已生成: {output_path}")
    
    def compile_pdf(self, latex_path: str):
        """
        编译LaTeX为PDF
        
        Args:
            latex_path: LaTeX文件路径
        """
        # 这里需要调用系统LaTeX编译器
        # 示例实现
        print(f"编译PDF: {latex_path}")
        print("请使用LaTeX编译器手动编译，或安装pdflatex")

## 第7部分：智能体定义

In [ ]:
class MathModelAgent:
    """数学建模智能体"""
    
    def __init__(self):
        """初始化数学建模智能体"""
        # 初始化组件
        self.knowledge_base = RAGKnowledgeBase()
        self.model_scheduler = ModelScheduler()
        self.hil = HILCollaboration()
        self.web_search = WebSearch()
        self.latex_generator = LatexGenerator()
        self.code_executor = CodeExecutor(timeout=60)
        
        # 初始化LLM
        self.llm = HelloAgentsLLM()
        
        # 定义系统提示词
        self.system_prompt = """你是一个专业的数学建模助手。你的任务是:
        1. 分析数学建模问题
        2. 推荐合适的数学模型
        3. 指导代码实现
        4. 辅助论文写作
        
        请以专业、清晰的方式回答问题。"""
        
        # 创建智能体
        self.agent = SimpleAgent(
            name="数学建模助手",
            llm=self.llm,
            system_prompt=self.system_prompt
        )
    
    def analyze_problem(self, problem_description: str) -> str:
        """
        分析数学建模问题
        
        Args:
            problem_description: 问题描述
        
        Returns:
            分析结果
        """
        # 从知识库检索相关知识
        knowledge = self.knowledge_base.search(problem_description)
        
        # 构建提示词
        prompt = f"""
        请分析以下数学建模问题：
        
        {problem_description}
        
        相关知识：
        {chr(10).join(knowledge)}
        
        请提供：
        1. 问题类型
        2. 关键变量
        3. 约束条件
        4. 可能的模型方向
        """
        
        # 使用简单模型分析
        self.model_scheduler.select_model("simple")
        result = self.agent.run(prompt)
        
        return result
    
    def recommend_model(self, analysis_result: str) -> str:
        """
        推荐数学模型
        
        Args:
            analysis_result: 分析结果
        
        Returns:
            推荐的模型
        """
        prompt = f"""
        基于以下分析结果，推荐合适的数学模型：
        
        {analysis_result}
        
        请推荐2-3个模型，并说明：
        1. 模型名称
        2. 适用场景
        3. 优缺点
        4. 实现难度
        """
        
        # 使用中等模型推荐
        self.model_scheduler.select_model("medium")
        result = self.agent.run(prompt)
        
        return result
    
    def generate_solution(self, model_name: str, data_description: str) -> str:
        """
        生成求解代码
        
        Args:
            model_name: 模型名称
            data_description: 数据描述
        
        Returns:
            求解代码
        """
        prompt = f"""
        请为以下模型生成Python求解代码：
        
        模型: {model_name}
        数据: {data_description}
        
        请提供：
        1. 完整的Python代码
        2. 详细的注释
        3. 使用说明
        """
        
        # 使用复杂模型生成代码
        self.model_scheduler.select_model("complex")
        result = self.agent.run(prompt)
        
        return result
    
    def execute_code(self, code: str, context: Optional[Dict[str, Any]] = None) -> Dict[str, Any]:
        """
        执行Python代码
        
        Args:
            code: 要执行的Python代码
            context: 执行上下文
        
        Returns:
            执行结果字典
        """
        print("执行代码...")
        result = self.code_executor.execute(code, context)
        
        if result['success']:
            print("代码执行成功！")
            if result['stdout']:
                print(f"输出:\n{result['stdout']}")
            if result['figure']:
                print("图表已生成")
        else:
            print(f"代码执行失败: {result['stderr']}")
        
        return result
    
    def execute_code_with_fix(self, code: str, max_retries: int = 3) -> Dict[str, Any]:
        """
        执行代码并在失败时尝试修复
        
        Args:
            code: 要执行的Python代码
            max_retries: 最大重试次数
        
        Returns:
            执行结果字典
        """
        def fix_callback(code, error):
            """使用LLM修复代码"""
            prompt = f"""
            以下代码执行出错，请修复：
            
            代码：
            {code}
            
            错误信息：
            {error}
            
            请提供修复后的完整代码。
            """
            
            # 使用复杂模型修复代码
            self.model_scheduler.select_model("complex")
            fixed_code = self.agent.run(prompt)
            
            # 提取代码块
            if "```python" in fixed_code:
                fixed_code = fixed_code.split("```python")[1].split("```")[0]
            
            return fixed_code
        
        print(f"执行代码（最多重试{max_retries}次）...")
        result = self.code_executor.execute_with_fix(
            code, max_retries=max_retries, fix_callback=fix_callback
        )
        
        if result['success']:
            print(f"代码执行成功！（尝试{result.get('attempts', 1)}次）")
            if result['stdout']:
                print(f"输出:\n{result['stdout']}")
            if result['figure']:
                print("图表已生成")
        else:
            print(f"代码执行失败（尝试{result.get('attempts', 1)}次）: {result['stderr']}")
        
        return result
    
    def generate_paper(self, content: Dict[str, str], output_path: str):
        """
        生成论文
        
        Args:
            content: 论文内容
            output_path: 输出路径
        """
        self.latex_generator.generate(content, output_path)
    
    def run(self, problem_description: str):
        """
        运行完整的建模流程
        
        Args:
            problem_description: 问题描述
        """
        print("=== 数学建模助手 ===")
        print("\n1. 分析问题...")
        analysis = self.analyze_problem(problem_description)
        
        # HIL: 等待用户确认
        action = self.hil.pause_for_review("问题分析", analysis)
        if action == "abort":
            print("流程已中止")
            return
        
        print("\n2. 推荐模型...")
        model_recommendation = self.recommend_model(analysis)
        
        # HIL: 等待用户选择模型
        action = self.hil.pause_for_review("模型推荐", model_recommendation)
        if action == "abort":
            print("流程已中止")
            return
        
        print("\n3. 生成求解代码...")
        solution = self.generate_solution("选择的模型", "数据描述")
        
        # HIL: 等待用户确认代码
        action = self.hil.pause_for_review("求解代码", solution)
        if action == "abort":
            print("流程已中止")
            return
        
        # 执行代码
        print("\n4. 执行代码...")
        # 从solution中提取代码
        if "```python" in solution:
            code = solution.split("```python")[1].split("```")[0]
            execution_result = self.execute_code_with_fix(code)
        
        print("\n5. 生成论文...")
        paper_content = {
            "title": "数学建模论文",
            "author": "16deng",
            "abstract": "这是论文摘要",
            "problem_restatement": analysis,
            "problem_analysis": analysis,
            "model_assumptions": "模型假设",
            "symbol_description": "符号说明",
            "model_establishment": solution,
            "model_verification": "模型验证",
            "model_evaluation": "模型评价",
            "references": "参考文献"
        }
        
        self.generate_paper(paper_content, "output/paper.tex")
        
        print("\n=== 建模完成 ===")

## 第8部分：运行示例

In [ ]:
# 创建数学建模智能体
agent = MathModelAgent()

# 示例问题
problem = """
某公司需要优化其物流配送路线。公司有10个配送点，每个配送点有一定数量的货物需要配送。
配送车辆从仓库出发，需要将货物送到各个配送点，然后返回仓库。
目标是找到最短的配送路线，使得总配送距离最短。
"""

print("=== 示例：物流配送路线优化 ===")
print(f"问题描述：{problem}")
print("\n开始运行数学建模助手...")

# 运行智能体
# agent.run(problem)

## 第9部分：可行性检查

In [ ]:
class FeasibilityChecker:
    """可行性检查器"""
    
    def __init__(self):
        """初始化可行性检查器"""
        self.checks = {
            "technical": self.check_technical,
            "data": self.check_data,
            "time": self.check_time,
            "resource": self.check_resource
        }
    
    def check_technical(self, requirements: Dict[str, Any]) -> Dict[str, Any]:
        """
        检查技术可行性
        
        Args:
            requirements: 技术需求
        
        Returns:
            检查结果
        """
        # 示例实现
        return {
            "status": "pass",
            "message": "技术方案可行",
            "details": "所有依赖库均可安装"
        }
    
    def check_data(self, data_requirements: Dict[str, Any]) -> Dict[str, Any]:
        """
        检查数据可行性
        
        Args:
            data_requirements: 数据需求
        
        Returns:
            检查结果
        """
        # 示例实现
        return {
            "status": "pass",
            "message": "数据可用",
            "details": "数据格式正确，质量满足要求"
        }
    
    def check_time(self, time_estimate: Dict[str, Any]) -> Dict[str, Any]:
        """
        检查时间可行性
        
        Args:
            time_estimate: 时间估计
        
        Returns:
            检查结果
        """
        # 示例实现
        return {
            "status": "pass",
            "message": "时间充足",
            "details": "预计完成时间在截止日期前"
        }
    
    def check_resource(self, resource_requirements: Dict[str, Any]) -> Dict[str, Any]:
        """
        检查资源可行性
        
        Args:
            resource_requirements: 资源需求
        
        Returns:
            检查结果
        """
        # 示例实现
        return {
            "status": "pass",
            "message": "资源可用",
            "details": "计算资源和API配额充足"
        }
    
    def check_all(self, requirements: Dict[str, Any]) -> Dict[str, Any]:
        """
        执行所有可行性检查
        
        Args:
            requirements: 所有需求
        
        Returns:
            检查结果
        """
        results = {}
        
        for check_name, check_func in self.checks.items():
            if check_name in requirements:
                results[check_name] = check_func(requirements[check_name])
        
        # 总体评估
        all_pass = all(r["status"] == "pass" for r in results.values())
        results["overall"] = {
            "status": "pass" if all_pass else "fail",
            "message": "所有检查通过" if all_pass else "部分检查未通过"
        }
        
        return results

In [ ]:
# 示例：运行可行性检查
checker = FeasibilityChecker()

requirements = {
    "technical": {"libraries": ["hello-agents", "langchain", "faiss"]},
    "data": {"format": "csv", "size": "100MB"},
    "time": {"deadline": "2026-09-01", "estimated_days": 7},
    "resource": {"api_calls": 1000, "compute": "CPU"}
}

results = checker.check_all(requirements)
print("可行性检查结果：")
print(json.dumps(results, indent=2, ensure_ascii=False))

## 项目总结

### 实现的功能

- 多模型智能调度
- RAG知识库
- HIL人机协作
- 联网搜索
- LaTeX论文生成
- 可行性检查

### 遇到的挑战

- 多模型调度的实现
- RAG知识库的优化
- 人机协作的交互设计

### 未来改进方向

- 支持更多数学模型
- 优化RAG检索效果
- 增强可视化功能
- 支持多人协作